# Product title category classifier

This notebook builds a reusable model that suggests a category from a product title. We inspect and clean the provided catalog, compare text-classification approaches on a holdout set, inspect their errors, then train and save the selected model.

We use only the product title as input because that is the information available at the time a colleague enters a new product. Product IDs, merchant information, listing dates, ratings, views, and internal codes are not used as features.

## 1. Load and inspect the source data

The source file has inconsistent whitespace in some column names, so we trim the headers before checking required fields. First we look at representative rows, missing-value counts, and the original category names and counts.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "product_model.py").exists():
    PROJECT_DIR = PROJECT_DIR / "SINE (Machine learning)"
if not (PROJECT_DIR / "product_model.py").exists():
    raise FileNotFoundError("Could not find product_model.py. Open this notebook from the project folder.")
sys.path.insert(0, str(PROJECT_DIR))

from product_model import DATA_FILE, MODEL_FILE, load_products, make_candidate_models, make_final_model

csv_path = DATA_FILE
raw_data = pd.read_csv(csv_path)
raw_data.columns = raw_data.columns.str.strip()
required_columns = {"Product Title", "Category Label"}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {', '.join(sorted(missing_columns))}")

print(f"Source: {csv_path.name}")
print(f"Rows: {len(raw_data):,} | columns: {len(raw_data.columns)}")
display(raw_data.head())
print("Missing values in source columns:")
display(raw_data.isna().sum().sort_values(ascending=False).to_frame("missing values"))
print("Original category labels:")
display(raw_data["Category Label"].value_counts(dropna=False).to_frame("products"))

## 2. Clean titles and standardize category labels

We remove records without a usable title or known label and normalize capitalization/spacing. The source contains a few singular or differently capitalized labels (`fridge`, `CPU`, `Mobile Phone`); these are mapped to their matching plural categories. We do not guess labels for other unknown values. The shared `load_products` function applies the same preparation during notebook analysis and command-line training.

In [ ]:
products = load_products(csv_path)
removed_count = len(raw_data) - len(products)
class_counts = products["Category Label"].value_counts().rename_axis("category").to_frame("products")
class_counts["share (%)"] = (class_counts["products"] / len(products) * 100).round(2)

print(f"Usable labeled titles: {len(products):,} (removed {removed_count:,} rows)")
print(f"Canonical categories: {products['Category Label'].nunique()}")
display(class_counts)

ax = class_counts["products"].sort_values().plot(kind="barh", figsize=(9, 6), color="steelblue")
ax.set_title("Product count by category after cleaning")
ax.set_xlabel("Number of products")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

print("Title length (characters):")
display(products["Product Title"].str.len().describe().to_frame("characters"))
print(f"Exact unique titles: {products['Product Title'].nunique():,} of {len(products):,} rows")

## 3. Feature engineering and leakage-aware holdout

The title is free text. Word n-grams capture category terms and short phrases; character n-grams can capture model codes, spelling variations, and compact product-specific fragments. We compare a word-only Logistic Regression and Naive Bayes baseline with a Linear SVC using both word and character features.

Repeated product titles are kept in one partition using `StratifiedGroupKFold`. This reduces overly optimistic evaluation from seeing an identical title in both training and test data, while keeping category proportions reasonably balanced.

In [ ]:
X = products["Product Title"]
y = products["Category Label"]
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_indices, test_indices = next(splitter.split(X, y, groups=X))
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

assert set(X_train).isdisjoint(set(X_test)), "A title unexpectedly occurs in both partitions."
print(f"Training rows: {len(X_train):,} | holdout rows: {len(X_test):,}")
print(f"Exact title overlap: {len(set(X_train) & set(X_test))}")
print("Holdout category proportions (%):")
display((y_test.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("share (%)"))

## 4. Train and compare candidate models

Each candidate is trained only on the training partition and evaluated on the same untouched holdout. We report overall accuracy and macro F1. Macro F1 gives every category equal weight, which is more informative than accuracy alone when category frequencies differ.

In [ ]:
candidate_models = make_candidate_models()
fitted_models = {}
predictions = {}
metric_rows = []
category_names = sorted(y.unique())

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = prediction
    metric_rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, prediction),
            "macro_f1": f1_score(y_test, prediction, average="macro"),
            "weighted_f1": f1_score(y_test, prediction, average="weighted"),
        }
    )
    print(f"\n{name}\n" + "-" * len(name))
    print(classification_report(y_test, prediction, labels=category_names, zero_division=0))

results = pd.DataFrame(metric_rows).set_index("model").sort_values("macro_f1", ascending=False)
display(results.round(4))

## 5. Confusion matrices and metric comparison

A confusion matrix shows which categories are mistaken for one another. Each panel uses the same category order; rows are true labels and columns are predicted labels. The metric chart makes the accuracy/macro-F1 comparison easy to scan.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, len(predictions), figsize=(8 * len(predictions), 7))
for ax, (name, prediction) in zip(axes, predictions.items()):
    matrix = confusion_matrix(y_test, prediction, labels=category_names)
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=category_names,
        yticklabels=category_names,
        ax=ax,
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted category")
    ax.set_ylabel("True category")
    ax.tick_params(axis="x", labelrotation=60)
    ax.tick_params(axis="y", labelrotation=0)
fig.suptitle("Confusion matrices (same holdout and category order)", y=1.02)
fig.tight_layout()
plt.show()

results[["accuracy", "macro_f1", "weighted_f1"]].plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
plt.title("Candidate model comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Error analysis and model choice

We choose the model with the highest macro F1, then inspect its per-category scores and most frequent confusions. A high score is encouraging, but category mistakes can still affect product discovery, so this analysis should guide a human review before deployment.

In [ ]:
best_name = results["macro_f1"].idxmax()
best_prediction = predictions[best_name]
best_report = pd.DataFrame(
    classification_report(
        y_test,
        best_prediction,
        labels=category_names,
        output_dict=True,
        zero_division=0,
    )
).T
display(best_report.loc[category_names, ["precision", "recall", "f1-score", "support"]].round(3))

best_matrix = confusion_matrix(y_test, best_prediction, labels=category_names)
confusions = []
for true_index, true_category in enumerate(category_names):
    for predicted_index, predicted_category in enumerate(category_names):
        if true_index != predicted_index and best_matrix[true_index, predicted_index] > 0:
            confusions.append((best_matrix[true_index, predicted_index], true_category, predicted_category))
confusions.sort(reverse=True)
print(f"Selected model by macro F1: {best_name} ({results.loc[best_name, 'macro_f1']:.4f})")
print("Most common category confusions:")
for count, true_category, predicted_category in confusions[:10]:
    print(f"  {true_category} predicted as {predicted_category}: {count}")

print("\nCandidate comparison:")
for name, row in results.iterrows():
    print(f"  {name}: accuracy={row['accuracy']:.4f}, macro F1={row['macro_f1']:.4f}")

print(
    f"\nConclusion: {best_name} is the strongest candidate on this grouped holdout "
    f"because it has the highest macro F1 ({results.loc[best_name, 'macro_f1']:.4f}). "
    "Macro F1 is the primary selection metric because it values all product categories "
    "equally; accuracy and weighted F1 are supporting metrics. The word-plus-character "
    "model can use both category terms and product-code-like text patterns. The simpler "
    "word-only baselines provide useful comparisons but score lower in this experiment. "
    "Review the displayed per-category report and confusions before using predictions "
    "without human confirmation."
)

## 7. Train the final model on all labeled records

After selecting an approach using the holdout results, we fit the same word-and-character Linear SVC pipeline on all cleaned records and save it for the interactive script. The saved model is not used to report the holdout scores above; those scores came from models that did not see the holdout rows during fitting.

In [ ]:
import joblib

final_model = make_final_model()
final_model.fit(products["Product Title"], products["Category Label"])
joblib.dump(final_model, MODEL_FILE)
print(f"Final model trained on {len(products):,} cleaned records.")
print(f"Saved model: {MODEL_FILE.resolve()}")
print("Try: python predict_category.py")